In [1]:
import numpy as np
import matplotlib.pyplot as plt

def build_vocab(text):
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return chars, stoi, itos

def text_to_indices(text, stoi):
    return np.array([stoi[ch] for ch in text], dtype=np.int32)

def init_reservoir(n_in, n_res, density, spectral_radius, input_scale, seed):
    """
    Initialize reservoir weights and input weights

    Args:
        n_in: number of input units
        n_res: number of reservoir units
        density: density of reservoir weight matrix
        spectral_radius: desired spectral radius of reservoir weight matrix
        input_scale: scaling factor for input weights
        seed: random seed for reproducibility
    """

    rng = np.random.default_rng(seed)
    # Input weights include bias; shape: n_res x (1 + n_in)
    Win = (rng.standard_normal((n_res, 1 + n_in)).astype(np.float32)) * input_scale
    # Reservoir weights sparse-ish dense matrix
    W = np.zeros((n_res, n_res), dtype=np.float32)
    nnz = int(density * n_res * n_res) # number of non-zero entries
    idx_i = rng.integers(0, n_res, size=nnz) # row indices
    idx_j = rng.integers(0, n_res, size=nnz) # column indices
    vals = rng.standard_normal(nnz).astype(np.float32) # weights
    W[idx_i, idx_j] = vals
    # Scale to desired spectral radius
    eigvals = np.linalg.eigvals(W.astype(np.float64))
    sr = np.max(np.abs(eigvals)).real
    if sr > 0:
        W *= (spectral_radius / sr)
    return Win, W

def rls_init(n_feat, reg):
    # P ~ (1/reg) * I; larger 1/reg means weaker prior (more plastic)
    P = (1.0 / reg) * np.eye(n_feat, dtype=np.float32)
    return P

def run_esn_training(
    indices,
    vocab_size,
    Win, W,
    alpha=0.3,
    washout=100,
    reg=1.0,
    loss_chunk=1000,
    include_input_in_readout=True,
    seed=42,
):
    """
    Args:
        indices: array of integer indices representing the training text
        vocab_size: size of the vocabulary (number of unique characters)
        Win: input weight matrix (n_res x (1 + n_in))
        W: reservoir weight matrix (n_res x n_res)
        alpha: leaking rate
        washout: number of initial time steps to discard for reservoir settling
        reg: regularization parameter for RLS
        loss_chunk: number of steps to average loss over for reporting
        include_input_in_readout: whether to include input in readout features
        seed: random seed for reproducibility
    """
    rng = np.random.default_rng(seed)
    n_res = W.shape[0]
    n_in = vocab_size
    n_out = vocab_size

    # One-hot lookup
    I = np.eye(vocab_size, dtype=np.float32)

    # Reservoir state
    x = np.zeros(n_res, dtype=np.float32)

    # Feature vector z = [1, u, x] or [1, x]
    if include_input_in_readout:
        n_feat = 1 + n_in + n_res
    else:
        n_feat = 1 + n_res

    # Readout
    Wout = np.zeros((n_out, n_feat), dtype=np.float32)

    # RLS inverse correlation matrix
    P = rls_init(n_feat, reg=reg)

    # Loss tracking
    losses = []
    chunk_losses = []
    steps_seen = 0

    # Training loop (teacher forcing), using all pairs (t -> t+1) after washout
    # We will update P and Wout only after washout
    T = len(indices) - 1
    bias = np.array([1.0], dtype=np.float32)

    for t in range(T):
        if t % 1000 == 0 and t > 0:
            print(f"  Training step {t}/{T}")
        u_idx = indices[t]
        v_idx = indices[t + 1]

        u = I[u_idx]  # one-hot current
        v = I[v_idx]  # one-hot next

        # Reservoir update: x <- (1 - alpha) * x + alpha * tanh( Win @ [1; u] + W @ x )
        in_vec = np.concatenate([bias, u], dtype=np.float32)  # shape 1 + n_in
        preact = Win @ in_vec + W @ x
        x = (1.0 - alpha) * x + alpha * np.tanh(preact)

        # Feature vector
        if include_input_in_readout:
            z = np.concatenate([bias, u, x], dtype=np.float32)  # shape n_feat
        else:
            z = np.concatenate([bias, x], dtype=np.float32)

        # Predict
        y_pred = Wout @ z  # linear logits for each char

        # Compute MSE loss vs one-hot target
        err = v - y_pred
        mse = np.mean(err * err)
        # During washout, just track reservoir settling; no RLS updates
        if t >= washout:
            # RLS update
            # k = P z / (1 + z^T P z)
            Pz = P @ z
            denom = 1.0 + float(z @ Pz)
            k = Pz / denom  # shape (n_feat,)

            # Wout <- Wout + (v - y_pred) k^T
            # Broadcasting outer product for multi-output
            Wout += np.outer(err, k).astype(np.float32)

            # P <- P - k (z^T P)
            P -= np.outer(k, Pz).astype(np.float32)

            # record loss
            chunk_losses.append(mse)
            steps_seen += 1
            if steps_seen % loss_chunk == 0:
                mean_loss = float(np.mean(chunk_losses))
                losses.append(mean_loss)
                chunk_losses = []

    # If last partial chunk exists, record it
    if len(chunk_losses) > 0:
        losses.append(float(np.mean(chunk_losses)))

    return Wout, losses

def evaluate_accuracy(indices, vocab_size, Win, W, Wout, alpha=0.3, include_input_in_readout=True):
    # Teacher forcing evaluation: predict next char from current char and current state
    n_res = W.shape[0]
    I = np.eye(vocab_size, dtype=np.float32)
    x = np.zeros(n_res, dtype=np.float32)
    bias = np.array([1.0], dtype=np.float32)
    correct = 0
    total = 0
    T = len(indices) - 1
    for t in range(T):
        u_idx = indices[t]
        v_idx = indices[t + 1]
        u = I[u_idx]

        in_vec = np.concatenate([bias, u], dtype=np.float32)
        preact = Win @ in_vec + W @ x
        x = (1.0 - alpha) * x + alpha * np.tanh(preact)

        if include_input_in_readout:
            z = np.concatenate([bias, u, x], dtype=np.float32)
        else:
            z = np.concatenate([bias, x], dtype=np.float32)

        y_pred = Wout @ z
        pred_idx = int(np.argmax(y_pred))
        correct += (pred_idx == v_idx)
        total += 1
    acc = correct / total if total > 0 else 0.0
    return acc

def main_jupyter(train_ratio=0.9,
          n_res=300,
          density=0.05,
          spectral_radius=0.9,
          alpha=0.3,
          input_scale=1.0,
          reg=1.0,
          washout=100,
          loss_chunk=2000,
          include_input=True,
          seed=42):
    # Set parameters directly instead of using argparse
    params = {
        "path": "tinyshakespeare.txt",  # Make sure this file exists
        "train_ratio": train_ratio,
        "n_res": n_res,
        "density": density,
        "spectral_radius": spectral_radius,
        "alpha": alpha,
        "input_scale": input_scale,
        "reg": reg,
        "washout": washout,
        "loss_chunk": loss_chunk,
        "include_input": include_input,
        "seed": seed
    }
    
    # Load text
    try:
        with open(params["path"], "r", encoding="utf-8") as f:
            text = f.read()
    except FileNotFoundError:
        print(f"Error: File {params['path']} not found!")
        return
    
    # Build vocab and indices
    chars, stoi, itos = build_vocab(text)
    data = text_to_indices(text, stoi)
    vocab_size = len(chars)
    N = len(data)

    # train/test split
    split = int(params["train_ratio"] * N)
    train_idx = data[:split]
    test_idx  = data[split:]

    print(f"Text length: {N}, Vocab size: {vocab_size}")
    print(f"Train steps: {len(train_idx)-1}, Test steps: {len(test_idx)-1}")

    # Initialize reservoir
    Win, W = init_reservoir(
        n_in=vocab_size,
        n_res=params["n_res"],
        density=params["density"],
        spectral_radius=params["spectral_radius"],
        input_scale=params["input_scale"],
        seed=params["seed"],
    )

    # Train (RLS online)
    Wout, losses = run_esn_training(
        train_idx,
        vocab_size,
        Win, W,
        alpha=params["alpha"],
        washout=params["washout"],
        reg=params["reg"],
        loss_chunk=params["loss_chunk"],
        include_input_in_readout=params["include_input"],
        seed=params["seed"],
    )

    # Evaluate accuracy on test set
    test_acc = evaluate_accuracy(
        test_idx,
        vocab_size,
        Win, W, Wout,
        alpha=params["alpha"],
        include_input_in_readout=params["include_input"],
    )

    print(f"Test accuracy (next-char, teacher forcing): {test_acc:.4f}")

    # Plot training loss
    plt.figure(figsize=(8,4))
    xs = np.arange(1, len(losses)+1) * params["loss_chunk"]
    plt.plot(xs, losses, label="Training MSE (avg per chunk)")
    plt.xlabel("Training steps")
    plt.ylabel("MSE")
    plt.title("ESN Readout Online Training Loss (RLS)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig("training_loss.png", dpi=150)
    try:
        plt.show()
    except Exception:
        pass

    return test_acc

In [11]:
import random

def run_esn_with_params(params_dict):
    """
    Wrapper to run main_jupyter with given params and return accuracy.
    Assumes main_jupyter now returns test_acc.
    """
    # Print current parameters being tested
    print(f"\n--- Testing parameters ---")
    for key, value in params_dict.items():
        print(f"{key}: {value}")
    print("-------------------------\n")

    # Extract params (skip fixed ones like train_ratio, loss_chunk, seed)

    acc = main_jupyter(
        train_ratio=params_dict['train_ratio'],  # Fixed at 0.9
        n_res=int(params_dict['n_res']),
        density=params_dict['density'],
        spectral_radius=params_dict['spectral_radius'],
        alpha=params_dict['alpha'],
        input_scale=params_dict['input_scale'],
        reg=params_dict['reg'],
        washout=int(params_dict['washout']),
        loss_chunk=2000,  # Fixed
        include_input=params_dict['include_input'],
        seed=42  # Fixed for reproducibility
    )
    return acc

def create_individual(param_ranges):
    """Create a random individual (dict of params)"""
    individual = {}
    for param, (low, high) in param_ranges.items():
        if isinstance(low, bool):  # For booleans like include_input
            individual[param] = random.choice([True, False])
        elif isinstance(low, int):
            individual[param] = random.randint(low, high)
        else:
            individual[param] = random.uniform(low, high)
    individual['train_ratio'] = 0.9  # Fixed
    return individual

def fitness(individual):
    """Fitness: test accuracy (higher is better)"""
    try:
        acc = run_esn_with_params(individual)
        return acc  # Maximize accuracy
    except Exception as e:
        print(f"Error evaluating individual: {e}")
        return 0.0  # Penalty for invalid params

def tournament_selection(population, fitnesses, tournament_size=3):
    """Select parent via tournament"""
    selected = random.sample(list(zip(population, fitnesses)), tournament_size)
    return max(selected, key=lambda x: x[1])[0]

def crossover(parent1, parent2):
    """Simple blend crossover for continuous params, copy for discrete"""
    child = {}
    for param in parent1:
        if param in ['include_input', 'train_ratio']:
            # Copy bool/fixed
            child[param] = parent1[param] if random.random() < 0.5 else parent2[param]
        elif isinstance(parent1[param], int):
            # Average and round for ints
            child[param] = int((parent1[param] + parent2[param]) / 2)
        else:
            # Blend for floats
            alpha = random.uniform(0, 1)
            child[param] = alpha * parent1[param] + (1 - alpha) * parent2[param]
    return child

def mutate(individual, param_ranges, mutation_rate=0.1):
    """Mutate with probability"""
    for param, (low, high) in param_ranges.items():
        if random.random() < mutation_rate:
            if isinstance(low, bool):
                individual[param] = random.choice([True, False])
            elif isinstance(low, int):
                individual[param] = random.randint(low, high)
            else:
                individual[param] = random.uniform(low, high)
    return individual

def genetic_algorithm(pop_size=10, generations=5, param_ranges=None):
    """
    Simple GA to optimize ESN params.
    
    Args:
        pop_size: Population size (small due to ESN training cost)
        generations: Number of generations
        param_ranges: Dict of {param: (low, high)} for bounds
    
    Returns:
        Best individual and its fitness
    """
    if param_ranges is None:
        # Define tunable parameter ranges (based on typical ESN values)
        param_ranges = {
            'n_res': (350, 450),  # Reservoir size (int)
            'density': (0.005, 0.015),  # Keep sparse to avoid chaos
            'spectral_radius': (0.9, 1.1),  # For echo state property
            'alpha': (0.4, 0.7),  # Leaking rate
            'input_scale': (0.5, 1.5),  # Input drive
            'reg': (0.5, 2.0),  # RLS regularization
            'washout': (50, 200),  # Washout period (int)
            'include_input': (True, True)  # Boolean (True/False)
        }
    
    # Initialize population
    population = [create_individual(param_ranges) for _ in range(pop_size)]
    
    for gen in range(generations):
        print(f"\n--- Generation {gen + 1}/{generations} ---")
        
        # Evaluate fitness
        fitnesses = [fitness(ind) for ind in population]
        best_idx = np.argmax(fitnesses)
        best_acc = fitnesses[best_idx]
        print(f"Best so far: {best_acc:.4f} with params: {population[best_idx]}")
        
        # Create new population
        new_population = []
        for _ in range(pop_size):
            # Selection
            parent1 = tournament_selection(population, fitnesses)
            parent2 = tournament_selection(population, fitnesses)
            
            # Crossover
            child = crossover(parent1, parent2)
            
            # Mutation
            child = mutate(child, param_ranges)
            
            new_population.append(child)
        
        population = new_population
    
    # Final evaluation
    fitnesses = [fitness(ind) for ind in population]
    best_idx = np.argmax(fitnesses)
    best_individual = population[best_idx]
    best_fitness = fitnesses[best_idx]
    
    print(f"\n=== GA Complete ===")
    print(f"Best params: {best_individual}")
    print(f"Best test accuracy: {best_fitness:.4f}")
    
    return best_individual, best_fitness


In [ ]:
# Run the GA (this will take time - each eval trains an ESN!)
# Adjust pop_size/generations if too slow (e.g., pop_size=5, generations=3 for testing)
best_params, best_acc = genetic_algorithm(pop_size=10, generations=6)

# Optional: Re-run with best params to confirm
print("\nRe-running with best params for final plot...")
final_acc = run_esn_with_params(best_params)
print(f"Confirmed final accuracy: {final_acc:.4f}")

=== GA Complete ===
Best params: {'n_res': 446, 'density': 0.009198661972961044, 'spectral_radius': 0.9992008896184146, 'alpha': 0.6899951586152183, 'input_scale': 1.404935196668347, 'reg': 0.9700236304889065, 'washout': 166, 'include_input': True, 'train_ratio': 0.9}
Best test accuracy: 0.3802

Re-running with best params for final plot...

--- Testing parameters ---
n_res: 446
density: 0.009198661972961044
spectral_radius: 0.9992008896184146
alpha: 0.6899951586152183
input_scale: 1.404935196668347
reg: 0.9700236304889065
washout: 166
include_input: True
train_ratio: 0.9
-------------------------

Text length: 1115394, Vocab size: 65
Train steps: 1003853, Test steps: 111539
Test accuracy (next-char, teacher forcing): 0.3802

In [2]:
main_jupyter(
    train_ratio=0.9,
    n_res=int(100),
    density=0.001,
    spectral_radius=1.0,
    alpha=0.69,
    input_scale=1.4,
    reg=0.97,
    washout=166,
    loss_chunk=2000,
    include_input=True,
    seed=42
)

Text length: 1115394, Vocab size: 65
Train steps: 1003853, Test steps: 111539
  Training step 1000/1003853
  Training step 2000/1003853
  Training step 3000/1003853
  Training step 4000/1003853
  Training step 5000/1003853
  Training step 6000/1003853
  Training step 7000/1003853
  Training step 8000/1003853
  Training step 9000/1003853
  Training step 10000/1003853
  Training step 11000/1003853
  Training step 12000/1003853
  Training step 13000/1003853
  Training step 14000/1003853
  Training step 15000/1003853
  Training step 16000/1003853
  Training step 17000/1003853
  Training step 18000/1003853
  Training step 19000/1003853
  Training step 20000/1003853
  Training step 21000/1003853
  Training step 22000/1003853
  Training step 23000/1003853
  Training step 24000/1003853
  Training step 25000/1003853
  Training step 26000/1003853
  Training step 27000/1003853
  Training step 28000/1003853
  Training step 29000/1003853
  Training step 30000/1003853
  Training step 31000/1003853
 

KeyboardInterrupt: 